# NumPy Fundamentals

A hands-on walkthrough of the six core ideas:

1. What is NumPy and why we use it
2. Creating NumPy arrays
3. Array indexing and slicing
4. Array operations
5. Mathematical functions
6. Array reshaping



---
## 1. What is NumPy and why it is used

**NumPy** (Numerical Python) gives Python a new data type: the **ndarray** (n-dimensional array).

Why bother, when Python already has lists?

| Python list | NumPy array |
|---|---|
| Can mix types (`[1, "a", 3.5]`) | One single type -> stored compactly in memory |
| Maths needs a loop | Maths applies to the whole array at once (*vectorisation*) |
| Slow for big data | Written in C underneath -> much faster |
| No built-in 2D/3D maths | Matrices, statistics, linear algebra built in |

NumPy is the foundation that pandas, scikit-learn, TensorFlow and PyTorch are all built on,
so learning it first makes everything after it easier.

In [1]:
# The standard import. "np" is the nickname everyone uses - stick to it.
import numpy as np

print("NumPy version:", np.__version__)

NumPy version: 2.2.4


In [2]:
# THE PROBLEM WITH LISTS: maths does not do what you expect.
py_list = [1, 2, 3, 4]
print(py_list * 2)        # repeats the list instead of doubling the numbers
# print(py_list + 10)     # would raise a TypeError - you cannot add a number to a list

# THE NUMPY WAY: the operation is applied to every element (this is "vectorisation").
np_array = np.array([1, 2, 3, 4])
print(np_array * 2)       # [2 4 6 8]
print(np_array + 10)      # [11 12 13 14]

[1, 2, 3, 4, 1, 2, 3, 4]
[2 4 6 8]
[11 12 13 14]


In [3]:
# WHY IT MATTERS: speed. Sum a million numbers both ways and compare.
import time

data_list = list(range(1_000_000))
data_array = np.arange(1_000_000)

start = time.time()
sum(x * 2 for x in data_list)          # pure Python loop
list_time = time.time() - start

start = time.time()
data_array * 2                         # NumPy, no loop written by us
array_time = time.time() - start

print(f"Python list : {list_time:.4f} s")
print(f"NumPy array : {array_time:.4f} s")
print(f"NumPy is roughly {list_time / array_time:.0f}x faster here")

Python list : 0.2221 s
NumPy array : 0.0093 s
NumPy is roughly 24x faster here


---
## 2. Creating NumPy arrays

There are two families of ways to make an array:

* **From data you already have** - `np.array()` on a list.
* **Generated for you** - `zeros`, `ones`, `arange`, `linspace`, `random`, ... handy for
  placeholders and test data.

In [4]:
# From a Python list -> 1D array (a "vector")
a = np.array([10, 20, 30, 40, 50])
print(a)
print(type(a))            # <class 'numpy.ndarray'>

# From a list of lists -> 2D array (a "matrix"). Each inner list is one row.
b = np.array([[1, 2, 3],
              [4, 5, 6]])
print(b)

[10 20 30 40 50]
<class 'numpy.ndarray'>
[[1 2 3]
 [4 5 6]]


In [5]:
# Every array carries information about itself - these 4 attributes come up constantly.
print("b        =\n", b)
print("shape    :", b.shape)   # (rows, columns)
print("ndim     :", b.ndim)    # number of dimensions -> 2
print("size     :", b.size)    # total number of elements -> 6
print("dtype    :", b.dtype)   # the single data type of all elements -> int64/int32

b        =
 [[1 2 3]
 [4 5 6]]
shape    : (2, 3)
ndim     : 2
size     : 6
dtype    : int64


In [6]:
# Generated arrays - useful when you need a starting point of a known size.

print(np.zeros(5))              # five 0.0s   -> common as an empty container
print(np.ones((2, 3)))          # 2x3 of 1.0  -> shape is passed as a tuple
print(np.full((2, 2), 7))       # 2x2 filled with 7
print(np.eye(3))                # identity matrix: 1s on the diagonal

[0. 0. 0. 0. 0.]
[[1. 1. 1.]
 [1. 1. 1.]]
[[7 7]
 [7 7]]
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


In [7]:
# arange = "array range". Same idea as Python's range(start, stop, step); stop is excluded.
print(np.arange(0, 10, 2))      # [0 2 4 6 8]

# linspace = "linear space": give a start, a stop, and HOW MANY numbers you want.
# Unlike arange, the stop value IS included. Great for plotting smooth curves.
print(np.linspace(0, 1, 5))     # [0.   0.25 0.5  0.75 1.  ]

[0 2 4 6 8]
[0.   0.25 0.5  0.75 1.  ]


In [ ]:
# Random data - perfect for practising without downloading a dataset.
np.random.seed(42)              # fixes the randomness so you get the same numbers every time you run the code

print(np.random.rand(3))            # 3 floats between 0 and 1
print(np.random.randint(1, 100, 5)) # 5 whole numbers from 1 up to (not incl.) 100
print(np.random.randn(2, 3))        # 2x3 from the normal/bell curve (mean 0, std 1)

[0.37454012 0.95071431 0.73199394]
[61 21 83 87 75]
[[-2.01096289 -0.49280342  0.39257975]
 [-0.92918467  0.07983181 -0.1595165 ]]


In [9]:
# Choosing the dtype yourself: smaller types use less memory, floats allow decimals.
ints = np.array([1, 2, 3], dtype='int32')
floats = np.array([1, 2, 3], dtype='float64')

print(ints, ints.dtype)
print(floats, floats.dtype)

# astype() converts an existing array (it returns a NEW array, the original is untouched).
print(floats.astype('int32'))

[1 2 3] int32
[1. 2. 3.] float64
[1 2 3]


---
## 3. Array indexing and slicing

Indexing = pulling out **one** element. Slicing = pulling out a **range**.

The rules match Python lists (`start:stop:step`, stop excluded, negative index counts from
the end) with two additions:

* For 2D arrays you use **one bracket with a comma**: `arr[row, column]`.
* A slice gives you a **view**, not a copy - changing the slice changes the original.

In [10]:
arr = np.array([10, 20, 30, 40, 50, 60])

# Single elements - counting starts at 0
print(arr[0])      # first  -> 10
print(arr[3])      # fourth -> 40
print(arr[-1])     # last   -> 60

10
40
60


In [11]:
# Slicing: arr[start:stop:step]  (stop is NOT included)
print(arr[1:4])    # [20 30 40]
print(arr[:3])     # from the beginning -> [10 20 30]
print(arr[3:])     # to the end         -> [40 50 60]
print(arr[::2])    # every 2nd element  -> [10 30 50]
print(arr[::-1])   # reversed           -> [60 50 40 30 20 10]

[20 30 40]
[10 20 30]
[40 50 60]
[10 30 50]
[60 50 40 30 20 10]


In [12]:
# 2D indexing: arr2d[row, column]
arr2d = np.array([[1,  2,  3,  4],
                  [5,  6,  7,  8],
                  [9, 10, 11, 12]])

print(arr2d[0, 0])      # row 0, col 0 -> 1
print(arr2d[1, 2])      # row 1, col 2 -> 7
print(arr2d[-1, -1])    # bottom-right -> 12

1
7
12


In [13]:
# 2D slicing: the part before the comma slices ROWS, after it slices COLUMNS.
print("whole row 1     :", arr2d[1])         # or arr2d[1, :]
print("whole column 2  :", arr2d[:, 2])      # ":" means "all rows"
print("first 2 rows, last 2 cols:\n", arr2d[:2, 2:])

whole row 1     : [5 6 7 8]
whole column 2  : [ 3  7 11]
first 2 rows, last 2 cols:
 [[3 4]
 [7 8]]


In [14]:
# Boolean ("mask") indexing - the most useful trick in NumPy.
scores = np.array([45, 88, 72, 95, 60, 30])

mask = scores > 70          # a True/False array of the same shape
print(mask)
print(scores[mask])         # keeps only the elements where the mask is True

# Usually written in one line:
print(scores[scores > 70])

# Combine conditions with & (and) / | (or) - each condition needs its own brackets.
print(scores[(scores > 50) & (scores < 90)])

[False  True  True  True False False]
[88 72 95]
[88 72 95]
[88 72 60]


In [15]:
# Fancy indexing - pass a LIST of positions to grab them in any order you like.
print(scores[[0, 2, 4]])    # elements at index 0, 2 and 4

# Assigning through an index or a mask edits the array in place.
scores[scores < 50] = 0     # replace every fail with 0
print(scores)

[45 72 60]
[ 0 88 72 95 60  0]


In [16]:
# IMPORTANT: a slice is a VIEW into the same memory, not a copy.
original = np.array([1, 2, 3, 4, 5])
view = original[0:3]
view[0] = 999
print("original changed too:", original)

# Use .copy() when you want an independent array.
safe = original[0:3].copy()
safe[0] = -1
print("original untouched  :", original)

original changed too: [999   2   3   4   5]
original untouched  : [999   2   3   4   5]


---
## 4. Array operations

Operations work **element by element**, with no loop from you.

When two arrays have different shapes, NumPy tries to stretch the smaller one to match -
this is called **broadcasting** (e.g. adding a single number to a whole array).

In [17]:
x = np.array([1, 2, 3, 4])
y = np.array([10, 20, 30, 40])

print(x + y)     # element-wise addition
print(y - x)
print(x * y)     # element-wise multiply, NOT matrix multiplication
print(y / x)
print(x ** 2)    # each element squared
print(y % 3)     # remainder of each element

[11 22 33 44]
[ 9 18 27 36]
[ 10  40  90 160]
[10. 10. 10. 10.]
[ 1  4  9 16]
[1 2 0 1]


In [18]:
# Array + single number: the number is "broadcast" to every element.
print(x + 100)
print(x * 10)

# Comparisons also return an array (of True/False), element by element.
print(x > 2)
print(x == y / 10)

[101 102 103 104]
[10 20 30 40]
[False False  True  True]
[ True  True  True  True]


In [19]:
# Broadcasting between different shapes: a (3,1) column + a (3,) row -> (3,3) grid.
col = np.array([[1], [2], [3]])     # shape (3, 1)
row = np.array([10, 20, 30])        # shape (3,)

print(col + row)

[[11 21 31]
 [12 22 32]
 [13 23 33]]


In [20]:
# Aggregations squash an array down to a summary number.
data = np.array([[1, 2, 3],
                 [4, 5, 6]])

print("sum of everything :", data.sum())
print("column sums (down):", data.sum(axis=0))   # axis=0 -> collapse the rows
print("row sums (across) :", data.sum(axis=1))   # axis=1 -> collapse the columns
print("min / max         :", data.min(), data.max())
print("position of max   :", data.argmax())      # index in the flattened array

sum of everything : 21
column sums (down): [5 7 9]
row sums (across) : [ 6 15]
min / max         : 1 6
position of max   : 5


In [21]:
# Matrix multiplication is deliberately a different symbol: @ (or np.dot).
m1 = np.array([[1, 2],
               [3, 4]])
m2 = np.array([[5, 6],
               [7, 8]])

print("element-wise (*):\n", m1 * m2)
print("matrix product (@):\n", m1 @ m2)

element-wise (*):
 [[ 5 12]
 [21 32]]
matrix product (@):
 [[19 22]
 [43 50]]


---
## 5. Mathematical functions

NumPy ships with "universal functions" (**ufuncs**) - maths applied to every element at once.
They are the vectorised versions of Python's `math` module, which only works on one number.

In [22]:
v = np.array([1, 4, 9, 16, 25])

print("sqrt :", np.sqrt(v))        # square root of each element
print("exp  :", np.exp([0, 1, 2])) # e^x
print("log  :", np.log([1, np.e])) # natural log
print("log10:", np.log10([1, 10, 100]))
print("abs  :", np.abs([-3, -1, 2]))

sqrt : [1. 2. 3. 4. 5.]
exp  : [1.         2.71828183 7.3890561 ]
log  : [0. 1.]
log10: [0. 1. 2.]
abs  : [3 1 2]


In [23]:
# Trigonometry works in RADIANS, so convert from degrees first.
angles = np.array([0, 30, 45, 60, 90])
radians = np.radians(angles)

print("sin:", np.round(np.sin(radians), 3))   # round() keeps the output readable
print("cos:", np.round(np.cos(radians), 3))

sin: [0.    0.5   0.707 0.866 1.   ]
cos: [1.    0.866 0.707 0.5   0.   ]


In [24]:
# Rounding family
prices = np.array([2.345, 7.891, 3.5, -1.27])

print("round to 1dp :", np.round(prices, 1))
print("floor (down) :", np.floor(prices))
print("ceil  (up)   :", np.ceil(prices))

round to 1dp : [ 2.3  7.9  3.5 -1.3]
floor (down) : [ 2.  7.  3. -2.]
ceil  (up)   : [ 3.  8.  4. -1.]


In [25]:
# Statistics - the functions you will reach for in every ML project.
marks = np.array([55, 78, 92, 61, 78, 84])

print("mean   :", np.mean(marks))     # average
print("median :", np.median(marks))   # middle value, ignores extreme outliers
print("std    :", np.std(marks))      # spread around the mean
print("var    :", np.var(marks))      # std squared
print("min/max:", np.min(marks), np.max(marks))
print("sum    :", np.sum(marks))

mean   : 74.66666666666667
median : 78.0
std    : 12.801909579781013
var    : 163.88888888888889
min/max: 55 92
sum    : 448


In [26]:
# Statistics on a 2D array - remember axis=0 goes DOWN columns, axis=1 goes ACROSS rows.
table = np.array([[80, 70, 90],     # student 1
                  [60, 85, 75],     # student 2
                  [95, 88, 100]])   # student 3

print("average per subject (columns):", table.mean(axis=0))
print("average per student (rows)   :", table.mean(axis=1))

average per subject (columns): [78.33333333 81.         88.33333333]
average per student (rows)   : [80.         73.33333333 94.33333333]


---
## 6. Array reshaping

The **data** stays the same, only the **shape** changes. The only rule:
the number of elements must match (a 12-element array can be 3x4 or 2x6, never 5x3).

In [27]:
flat = np.arange(12)     # [0 1 2 ... 11]
print(flat, flat.shape)

# reshape(rows, columns) - 3 * 4 == 12, so this is allowed.
grid = flat.reshape(3, 4)
print(grid)
print("new shape:", grid.shape)

[ 0  1  2  3  4  5  6  7  8  9 10 11] (12,)
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
new shape: (3, 4)


In [28]:
# -1 means "work this dimension out for me" - handy when you only care about one side.
print(flat.reshape(2, -1))    # 2 rows, NumPy computes 6 columns
print(flat.reshape(-1, 3))    # 3 columns, NumPy computes 4 rows

[[ 0  1  2  3  4  5]
 [ 6  7  8  9 10 11]]
[[ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]]


In [29]:
# Going back to 1D
print(grid.ravel())    # a view when possible -> fast, but linked to the original
print(grid.flatten())  # always a fresh copy -> safe to edit

[ 0  1  2  3  4  5  6  7  8  9 10 11]
[ 0  1  2  3  4  5  6  7  8  9 10 11]


In [30]:
# Transpose: swap rows and columns. Written .T for short.
print("original 3x4:\n", grid)
print("transposed 4x3:\n", grid.T)
print(grid.shape, "->", grid.T.shape)

original 3x4:
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
transposed 4x3:
 [[ 0  4  8]
 [ 1  5  9]
 [ 2  6 10]
 [ 3  7 11]]
(3, 4) -> (4, 3)


In [31]:
# Adding a dimension - many ML libraries insist on 2D input, even for a single row.
one_d = np.array([1, 2, 3])
print(one_d.shape)                    # (3,)

print(one_d.reshape(1, -1).shape)     # (1, 3) -> one row
print(one_d.reshape(-1, 1).shape)     # (3, 1) -> one column
print(one_d[:, np.newaxis].shape)     # same as above, np.newaxis inserts an axis

(3,)
(1, 3)
(3, 1)
(3, 1)


In [32]:
# Reshaping 3D: images are often (height, width, colour_channels).
cube = np.arange(24).reshape(2, 3, 4)   # 2 blocks of 3x4
print("shape:", cube.shape, "| ndim:", cube.ndim)
print(cube[0])                           # the first block

# Flatten a batch of images into rows of features - a very common ML step.
print("flattened per block:", cube.reshape(2, -1).shape)   # (2, 12)

shape: (2, 3, 4) | ndim: 3
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
flattened per block: (2, 12)


---
## Quick recap

| Task | Code |
|---|---|
| Make an array | `np.array([1,2,3])`, `np.zeros`, `np.arange`, `np.linspace` |
| Inspect it | `.shape`, `.ndim`, `.size`, `.dtype` |
| Get elements | `a[2]`, `a[1:4]`, `a2d[row, col]`, `a[a > 5]` |
| Do maths | `a + b`, `a * 2`, `a @ b` |
| Summarise | `.sum()`, `.mean()`, `.std()`, `axis=0` / `axis=1` |
| Change shape | `.reshape(r, c)`, `.T`, `.flatten()`, `-1` |

Next up: **pandas**, which puts labels (column names) on top of these arrays.